# Reto Semanal 3: Pipeline Neuronal para Noticias Falsas

- González López Josue Aldebaran

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path("/content/drive/MyDrive/ModeladoPredictivo2026")
sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import (
    PAD_IDX,
    UNK_IDX,
    build_vocabulary,
    text_to_indices,
    load_glove,
    FakeNewsDataset,
    create_dataloaders,
)

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"Proyecto : {PROJECT_ROOT}")

PyTorch  : 2.11.0+cu128
CUDA     : True
Proyecto : /content/drive/MyDrive/ModeladoPredictivo2026


In [3]:
# Intentar cargar datos limpios; si no existen, cargar el dataset original
DATA_DIR = PROJECT_ROOT / "data"

clean_path = DATA_DIR / "processed" / "wellfake_clean.csv" #ERROR FORZADO
raw_path = DATA_DIR / "raw" / "WELFake_Dataset.csv"

if clean_path.exists():
    df = pd.read_csv(clean_path)
    print(f"Datos limpios cargados desde: {clean_path}")
else:
    df = pd.read_csv(raw_path)
    # Limpieza básica si no hay datos preprocesados
    df = df.dropna(subset=["text", "label"])
    df["text"] = df["text"].astype(str).str.lower().str.strip()
    print(f"Datos originales cargados desde: {raw_path}")
    print("  (Ejecuta el Notebook 02 para obtener los datos limpios)")

print(f"\nForma del DataFrame: {df.shape}")
print(f"Columnas: {list(df.columns)}")
print(f"\nDistribución de etiquetas:")
print(df["label"].value_counts())
df.head(3)

Datos limpios cargados desde: /content/drive/MyDrive/ModeladoPredictivo2026/data/processed/wellfake_clean.csv

Forma del DataFrame: (61394, 5)
Columnas: ['title', 'text', 'label', 'title_len', 'text_len']

Distribución de etiquetas:
label
0    34237
1    27157
Name: count, dtype: int64


,title,text,label,title_len,text_len
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1,18,871
1,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1,18,34
2,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0,16,1321


In [4]:
from sklearn.model_selection import train_test_split

# Primera división: 80% train, 20% temporal
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])

# Segunda división: del 20% temporal → 10% val + 10% test
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])

print(f"Entrenamiento: {len(train_df):,} muestras ({len(train_df)/len(df)*100:.1f}%)")
print(f"Validación:    {len(val_df):,} muestras ({len(val_df)/len(df)*100:.1f}%)")
print(f"Prueba:        {len(test_df):,} muestras ({len(test_df)/len(df)*100:.1f}%)")

# Extraer listas de textos y etiquetas
train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()
val_texts = val_df["text"].tolist()
val_labels = val_df["label"].tolist()
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

Entrenamiento: 49,115 muestras (80.0%)
Validación:    6,139 muestras (10.0%)
Prueba:        6,140 muestras (10.0%)


In [5]:
# Construir vocabulario SOLO con datos de entrenamiento
MAX_VOCAB = 20_000
word2idx = build_vocabulary(train_texts, max_vocab=MAX_VOCAB)

print(f"Tamaño del vocabulario: {len(word2idx):,} palabras")
print(f"  (incluye <PAD> en índice {PAD_IDX} y <UNK> en índice {UNK_IDX})")

# Mostrar algunas palabras y sus índices
print(f"\nPrimeras 15 entradas del vocabulario:")
for i, (word, idx) in enumerate(list(word2idx.items())[:15]):
    print(f"  '{word}' → {idx}")

# Mostrar también algunas palabras comunes
print(f"\nEjemplos de palabras comunes:")
for word in ["the", "news", "trump", "said", "president", "government", "people"]:
    idx = word2idx.get(word, "NO ENCONTRADA")
    print(f"  '{word}' → {idx}")

Tamaño del vocabulario: 20,002 palabras
  (incluye <PAD> en índice 0 y <UNK> en índice 1)

Primeras 15 entradas del vocabulario:
  '<PAD>' → 0
  '<UNK>' → 1
  'the' → 2
  'to' → 3
  'of' → 4
  'and' → 5
  'a' → 6
  'in' → 7
  'that' → 8
  'on' → 9
  'is' → 10
  'for' → 11
  'he' → 12
  'with' → 13
  'was' → 14

Ejemplos de palabras comunes:
  'the' → 2
  'news' → 118
  'trump' → 28
  'said' → 25
  'president' → 53
  'government' → 96
  'people' → 51


In [6]:
# Crear datasets de PyTorch
MAX_LEN = 200

train_dataset = FakeNewsDataset(train_texts, train_labels, word2idx, max_len=MAX_LEN)
val_dataset = FakeNewsDataset(val_texts, val_labels, word2idx, max_len=MAX_LEN)
test_dataset = FakeNewsDataset(test_texts, test_labels, word2idx, max_len=MAX_LEN)

print(f"Datasets creados:")
print(f"  Train:  {len(train_dataset):,} muestras")
print(f"  Val:    {len(val_dataset):,} muestras")
print(f"  Test:   {len(test_dataset):,} muestras")

# Inspeccionar un elemento individual
sample_text, sample_label = train_dataset[0]
print(f"\nEjemplo del dataset (elemento 0):")
print(f"  Tipo texto:     {type(sample_text)} → {sample_text.dtype}")
print(f"  Forma texto:    {sample_text.shape}  (longitud fija = {MAX_LEN})")
print(f"  Tipo etiqueta:  {type(sample_label)} → {sample_label.dtype}")
print(f"  Valor etiqueta: {sample_label.item()}")
print(f"  Primeros 20 índices: {sample_text[:20].tolist()}")

Datasets creados:
  Train:  49,115 muestras
  Val:    6,139 muestras
  Test:   6,140 muestras

Ejemplo del dataset (elemento 0):
  Tipo texto:     <class 'torch.Tensor'> → torch.int64
  Forma texto:    torch.Size([200])  (longitud fija = 200)
  Tipo etiqueta:  <class 'torch.Tensor'> → torch.float32
  Valor etiqueta: 0.0
  Primeros 20 índices: [156, 152, 100, 6651, 1, 10, 307, 3, 1, 1932, 23, 85, 28, 7, 49, 4484, 2031, 11, 2, 66]


In [7]:
# Crear DataLoaders
BATCH_SIZE = 64

train_loader, val_loader, test_loader = create_dataloaders(
    train_data=(train_texts, train_labels),
    val_data=(val_texts, val_labels),
    test_data=(test_texts, test_labels),
    word2idx=word2idx,
    batch_size=BATCH_SIZE,
    max_len=MAX_LEN,
)

# Iterar un batch para verificar dimensiones
batch_texts, batch_labels = next(iter(train_loader))

print(f"\n=== Inspección de un batch ===")
print(f"Forma del batch de textos:    {batch_texts.shape}  → (batch_size, max_len)")
print(f"Forma del batch de etiquetas: {batch_labels.shape}  → (batch_size,)")
print(f"Dtype textos:    {batch_texts.dtype}")
print(f"Dtype etiquetas: {batch_labels.dtype}")
print(f"\nPrimer texto del batch (primeros 30 índices):")
print(f"  {batch_texts[0, :30].tolist()}")
print(f"Primeras 10 etiquetas del batch:")
print(f"  {batch_labels[:10].tolist()}")

DataLoaders creados:
  Entrenamiento: 49,115 muestras -> 768 batches
  Validacion:    6,139 muestras -> 96 batches
  Prueba:        6,140 muestras -> 96 batches
  Batch size: 64 | Max len: 200

=== Inspección de un batch ===
Forma del batch de textos:    torch.Size([64, 200])  → (batch_size, max_len)
Forma del batch de etiquetas: torch.Size([64])  → (batch_size,)
Dtype textos:    torch.int64
Dtype etiquetas: torch.float32

Primer texto del batch (primeros 30 índices):
  [16, 27, 1, 1, 1, 10, 2, 4670, 1, 37, 58, 953, 46, 14, 57, 46, 233, 2, 12649, 116, 2245, 31, 27, 24, 2975, 8, 46, 1861, 919, 108]
Primeras 10 etiquetas del batch:
  [1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0]


In [8]:
# Ruta al archivo GloVe — ajustar si lo guardaste en otra ubicación
EMBED_DIM = 50
GLOVE_PATH = PROJECT_ROOT / "data" / "embeddings" / "glove.6B.50d.txt"

if GLOVE_PATH.exists():
    # Cargar embeddings preentrenados
    embedding_matrix = load_glove(str(GLOVE_PATH), word2idx, embed_dim=EMBED_DIM)

    print(f"\nForma de la matriz de embeddings: {embedding_matrix.shape}")
    print(f"  → {embedding_matrix.shape[0]:,} palabras x {embedding_matrix.shape[1]} dimensiones")
    print(f"Dtype: {embedding_matrix.dtype}")

    # Verificar que el vector de PAD es ceros
    print(f"\nVector de <PAD> (debe ser ceros): {embedding_matrix[PAD_IDX][:10]}")

    # Mostrar vector de una palabra conocida
    word_ejemplo = "the"
    if word_ejemplo in word2idx:
        idx_ejemplo = word2idx[word_ejemplo]
        print(f"Vector de '{word_ejemplo}' (primeros 10 dims): {embedding_matrix[idx_ejemplo][:10].round(4)}")
else:
    print(f"Archivo GloVe no encontrado en: {GLOVE_PATH}")
    print("Sigue las instrucciones de la celda anterior para descargarlo.")
    print("\nCreando matriz de embeddings aleatoria como alternativa temporal...")
    embedding_matrix = np.random.normal(scale=0.6, size=(len(word2idx), EMBED_DIM))
    embedding_matrix[PAD_IDX] = np.zeros(EMBED_DIM)
    print(f"Forma de la matriz (aleatoria): {embedding_matrix.shape}")

GloVe cargado: 13,661 / 20,002 palabras encontradas (68.3% de cobertura)

Forma de la matriz de embeddings: (20002, 50)
  → 20,002 palabras x 50 dimensiones
Dtype: float64

Vector de <PAD> (debe ser ceros): [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Vector de 'the' (primeros 10 dims): [ 0.418   0.2497 -0.4124  0.1217  0.3453 -0.0445 -0.4969 -0.1786 -0.0007
 -0.6566]


In [9]:
# Crear la capa nn.Embedding e inicializarla con GloVe
vocab_size = len(word2idx)

embedding_layer = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=EMBED_DIM,
    padding_idx=PAD_IDX,  # Indica a PyTorch que el índice 0 es padding (gradiente=0)
)

# Cargar los pesos de GloVe en la capa
embedding_layer.weight = nn.Parameter(
    torch.tensor(embedding_matrix, dtype=torch.float32)
)

print(f"Capa nn.Embedding creada:")
print(f"  Forma de pesos: {embedding_layer.weight.shape}")
print(f"  padding_idx: {embedding_layer.padding_idx}")

# Verificar que los pesos se cargaron correctamente
# Comparar el vector de "the" en la matriz original vs. en la capa
word_test = "the"
if word_test in word2idx:
    idx_test = word2idx[word_test]

    # Vector de la matriz numpy original
    vec_numpy = embedding_matrix[idx_test][:5]

    # Vector de la capa de PyTorch
    with torch.no_grad():
        input_tensor = torch.tensor([idx_test], dtype=torch.long)
        vec_pytorch = embedding_layer(input_tensor).squeeze()[:5].numpy()

    print(f"\nVerificación para '{word_test}' (primeros 5 dims):")
    print(f"  Matriz numpy:   {vec_numpy.round(4)}")
    print(f"  nn.Embedding:   {vec_pytorch.round(4)}")
    print(f"  ¿Son iguales?   {np.allclose(vec_numpy, vec_pytorch)}")

# Verificar que PAD sigue siendo ceros
with torch.no_grad():
    pad_vec = embedding_layer(torch.tensor([PAD_IDX])).squeeze()
    print(f"\nVector de PAD (debe ser ceros): {pad_vec[:10].tolist()}")

Capa nn.Embedding creada:
  Forma de pesos: torch.Size([20002, 50])
  padding_idx: 0

Verificación para 'the' (primeros 5 dims):
  Matriz numpy:   [ 0.418   0.2497 -0.4124  0.1217  0.3453]
  nn.Embedding:   [ 0.418   0.2497 -0.4124  0.1217  0.3453]
  ¿Son iguales?   True

Vector de PAD (debe ser ceros): [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [10]:
# Calcular estadísticas de cobertura
# Una palabra "tiene GloVe" si su vector NO es aleatorio.
# Estrategia: comparar norma del vector — los vectores GloVe reales tienen
# características estadísticas diferentes a los aleatorios.

# Método directo: leer GloVe de nuevo y contar coincidencias
glove_words = set()

if GLOVE_PATH.exists():
    with open(str(GLOVE_PATH), "r", encoding="utf-8") as f:
        for line in f:
            word = line.split()[0]
            if word in word2idx:
                glove_words.add(word)

    total_vocab = len(word2idx)
    con_glove = len(glove_words)
    sin_glove = total_vocab - con_glove

    print(f"=== Estadísticas de cobertura GloVe ===\n")
    print(f"Tamaño del vocabulario:             {total_vocab:,}")
    print(f"Palabras CON vector GloVe:          {con_glove:,} ({con_glove/total_vocab*100:.1f}%)")
    print(f"Palabras SIN vector GloVe (random): {sin_glove:,} ({sin_glove/total_vocab*100:.1f}%)")

    # Mostrar algunas palabras sin GloVe
    palabras_sin_glove = [w for w in word2idx if w not in glove_words and w not in {"<PAD>", "<UNK>"}]
    if palabras_sin_glove:
        print(f"\nEjemplos de palabras sin vector GloVe (primeras 20):")
        for w in palabras_sin_glove[:20]:
            print(f"  - '{w}'")
        print(f"\n  (Estas palabras suelen ser: errores tipográficos, jerga de internet,")
        print(f"   nombres propios poco comunes, o tokens que resultaron de la limpieza)")
else:
    print("GloVe no disponible — no se pueden calcular estadísticas de cobertura.")
    print("Las estadísticas se mostrarán cuando descargues el archivo GloVe.")

=== Estadísticas de cobertura GloVe ===

Tamaño del vocabulario:             20,002
Palabras CON vector GloVe:          13,661 (68.3%)
Palabras SIN vector GloVe (random): 6,341 (31.7%)

Ejemplos de palabras sin vector GloVe (primeras 20):
  - 'trump’s'
  - '(reuters)'
  - 'it’s'
  - 'said,'
  - '“the'
  - '“i'
  - 'don’t'
  - 'it.'
  - 'trump,'
  - 'however,'
  - '“we'
  - 'that’s'
  - 'year,'
  - 'it,'
  - 'trump.'
  - 'that,'
  - 'them.'
  - 'clinton’s'
  - 'time,'
  - 'states,'

  (Estas palabras suelen ser: errores tipográficos, jerga de internet,
   nombres propios poco comunes, o tokens que resultaron de la limpieza)


# Mejora

In [11]:
!pip install contractions -q
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 57.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [12]:
import re, unicodedata
import contractions
import spacy

nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
POS_KEEP = {'NOUN', 'PROPN', 'VERB', 'ADJ', 'ADV'}

print(f'spaCy : {spacy.__version__}')

✅  Dependencias listas
spaCy : 3.8.14


In [13]:
def calcular_cobertura(texts, label=''):
    from sklearn.model_selection import train_test_split
    train_t, _ = train_test_split(texts, test_size=0.20, random_state=42)
    w2i = build_vocabulary(train_t, max_vocab=20_000)
    glove_w = set()
    with open(str(GLOVE_PATH), 'r', encoding='utf-8') as f:
        for line in f:
            w = line.split()[0]
            if w in w2i:
                glove_w.add(w)
    total = len(w2i)
    con   = len(glove_w)
    sin   = [w for w in w2i if w not in glove_w and w not in {'<PAD>', '<UNK>'}]
    print("="*50)
    print(f'  {label}')
    print(f'  Vocabulario : {total:,}')
    print(f'  Con GloVe   : {con:,}  ({con/total*100:.1f} %)')
    print(f'  Sin GloVe   : {total-con:,}  ({(total-con)/total*100:.1f} %)')
    print(f'  Top 10 sin cobertura: {sin[:10]}')
    return con/total*100

textos_base = df['text'].astype(str).str.lower().str.strip().tolist()
pct_base = calcular_cobertura(textos_base, 'BASELINE (lowercase + strip)')

  BASELINE (lowercase + strip)
  Vocabulario : 20,002
  Con GloVe   : 13,672  (68.4 %)
  Sin GloVe   : 6,330  (31.6 %)
  Top 10 sin cobertura: ['trump’s', '(reuters)', 'it’s', 'said,', '“the', '“i', 'don’t', 'it.', 'trump,', 'however,']


In [22]:
def paso_contracciones(text: str) -> str:
    try:
        text = contractions.fix(text)
    except IndexError:
        pass
    return text.lower().strip()

In [23]:
mask = df['text'].astype(str).str.contains("n't|'re|'ve|'ll", na=False)
for orig in df[mask]['text'].head(3):
    print(f'  ORIG : {str(orig)[:100]}')
    print(f'  P1   : {paso_contracciones(str(orig))[:100]}')
    print()

textos_p1 = df['text'].astype(str).apply(paso_contracciones).tolist()
pct_p1 = calcular_cobertura(textos_p1, 'PASO 1 — + Contracciones')
print(f'\nMejora vs baseline: +{pct_p1 - pct_base:.1f} pp')

  ORIG : FILE – In this Sept. 15, 2005 file photo, the marker that welcomes commuters to Cushing, Okla. is se
  P1   : file – in this sept. 15, 2005 file photo, the marker that welcomes commuters to cushing, okla. is se

  ORIG : The most punchable Alt-Right Nazi on the internet just got a thorough beatdown from Sen. Ben Sasse (
  P1   : the most punchable alt-right nazi on the internet just got a thorough beatdown from sen. ben sasse (

  ORIG : The NRA has a new favorite toy, but there are no bullets involved   except maybe the bullet points t
  P1   : the nra has a new favorite toy, but there are no bullets involved   except maybe the bullet points t

  PASO 1 — + Contracciones
  Vocabulario : 20,002
  Con GloVe   : 13,706  (68.5 %)
  Sin GloVe   : 6,296  (31.5 %)
  Top 10 sin cobertura: ['you.s.', 'trump’s', '(reuters)', '“i', 'said,', '“the', 'it.', '“we', 'trump,', 'however,']

Mejora vs baseline: +0.2 pp


In [24]:
textos_p1_list = df['text'].astype(str).apply(paso_contracciones).tolist()
textos_p2 = []
for doc in nlp.pipe(textos_p1_list, batch_size=512):
    tokens = [
        token.lemma_
        for token in doc
        if token.pos_ in POS_KEEP
        and not token.is_space
        and len(token.lemma_) > 1
    ]
    textos_p2.append(' '.join(tokens))

print('\u2705  Lematización completa\n')

✅  Lematización completa



In [25]:
for orig, p2 in zip(df['text'].head(3), textos_p2[:3]):
    print(f'  ORIG : {str(orig)[:100]}')
    print(f'  P2   : {p2[:100]}')
    print()

pct_p2 = calcular_cobertura(textos_p2, 'PASO 2 — + Lematización + POS')
print(f'\nMejora vs baseline: +{pct_p2 - pct_base:.1f} pp')
print(f'Mejora vs Paso 1  : +{pct_p2 - pct_p1:.1f} pp')

  ORIG : No comment is expected from Barack Obama Members of the #FYF911 or #FukYoFlag and #BlackLivesMatter 
  P2   : comment expect barack obama member fyf911 fukyoflag blacklivesmatt movement call lynching hanging wh

  ORIG :  Now, most of the demonstrators gathered last night were exercising their constitutional and protect
  P2   : now most demonstrator gather last night exercise constitutional protect right peaceful protest order

  ORIG : A dozen politically active pastors came here for a private dinner Friday night to hear a conversion 
  P2   : dozen politically active pastor come here private dinner friday night hear conversion story unique c

  PASO 2 — + Lematización + POS
  Vocabulario : 20,002
  Con GloVe   : 19,211  (96.0 %)
  Sin GloVe   : 791  (4.0 %)
  Top 10 sin cobertura: ['you.s', 'you.n', 'brexit', 'wikileak', '@realdonaldtrump', 'что', 'pende', 'daca', 'reince', 'daesh']

Mejora vs baseline: +27.7 pp
Mejora vs Paso 2  : +27.5 pp


In [26]:
def paso_signos(text: str) -> str:
    # URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Menciones y hashtags
    text = re.sub(r'[@#]\w+', '', text)
    # Puntuación residual
    text = re.sub(r'[^\w\s]', ' ', text)
    # Dígitos aislados
    text = re.sub(r'\b\d+\b', '', text)
    # Espacios múltiples
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [27]:
textos_final = [paso_signos(t) for t in textos_p2]

print('=== Original vs Pipeline Completo ===\n')
for orig, final in zip(df['text'].head(3), textos_final[:3]):
    print(f'  ORIG  : {str(orig)[:120]}')
    print(f'  FINAL : {final[:120]}')
    print()

pct_final = calcular_cobertura(textos_final, 'PIPELINE COMPLETO (P1+P2+P3+P4)')
print(f'\n{"="*50}')
print(f'MEJORA TOTAL vs baseline: +{pct_final - pct_base:.1f} pp')
print(f'{pct_base:.1f} %  ->  {pct_final:.1f} %')

=== Original vs Pipeline Completo ===

  ORIG  : No comment is expected from Barack Obama Members of the #FYF911 or #FukYoFlag and #BlackLivesMatter movements called for
  FINAL : comment expect barack obama member fyf911 fukyoflag blacklivesmatt movement call lynching hanging white people cop encou

  ORIG  :  Now, most of the demonstrators gathered last night were exercising their constitutional and protected right to peaceful
  FINAL : now most demonstrator gather last night exercise constitutional protect right peaceful protest order raise issue create 

  ORIG  : A dozen politically active pastors came here for a private dinner Friday night to hear a conversion story unique in the 
  FINAL : dozen politically active pastor come here private dinner friday night hear conversion story unique context presidential 

  PIPELINE COMPLETO (P1+P2+P3+P4)
  Vocabulario : 20,002
  Con GloVe   : 19,357  (96.8 %)
  Sin GloVe   : 645  (3.2 %)
  Top 10 sin cobertura: ['wikileak', 'brexit', 'что',

In [28]:
# Split con texto mejorado
df['text'] = textos_final

train_df2, temp_df2 = train_test_split(df, test_size=0.20, random_state=42, stratify=df['label'])
val_df2,  test_df2  = train_test_split(temp_df2, test_size=0.50, random_state=42, stratify=temp_df2['label'])

train_texts2 = train_df2['text'].tolist()
val_texts2   = val_df2['text'].tolist()
test_texts2  = test_df2['text'].tolist()
train_labels2 = train_df2['label'].tolist()
val_labels2   = val_df2['label'].tolist()
test_labels2  = test_df2['label'].tolist()

# Vocabulario SOLO con train
word2idx2 = build_vocabulary(train_texts2, max_vocab=20_000)
print(f'Vocabulario mejorado: {len(word2idx2):,} palabras')

Vocabulario mejorado: 20,002 palabras


In [29]:
# DataLoaders
train_loader2, val_loader2, test_loader2 = create_dataloaders(
    train_data=(train_texts2, train_labels2),
    val_data  =(val_texts2,   val_labels2),
    test_data =(test_texts2,  test_labels2),
    word2idx  =word2idx2,
    batch_size=BATCH_SIZE,
    max_len   =MAX_LEN,
)

batch_texts2, batch_labels2 = next(iter(train_loader2))

print(f'\n=== Inspección de un batch ===')
print(f'Forma del batch de textos:    {batch_texts2.shape}  → (batch_size, max_len)')
print(f'Forma del batch de etiquetas: {batch_labels2.shape}  → (batch_size,)')
print(f'Dtype textos:    {batch_texts2.dtype}')
print(f'Dtype etiquetas: {batch_labels2.dtype}')
print(f'\nPrimer texto del batch (primeros 30 índices):')
print(f'  {batch_texts2[0, :30].tolist()}')
print(f'Primeras 10 etiquetas del batch:')
print(f'  {batch_labels2[:10].tolist()}')

DataLoaders creados:
  Entrenamiento: 49,115 muestras -> 768 batches
  Validacion:    6,139 muestras -> 96 batches
  Prueba:        6,140 muestras -> 96 batches
  Batch size: 64 | Max len: 200

=== Inspección de un batch ===
Forma del batch de textos:    torch.Size([64, 200])  → (batch_size, max_len)
Forma del batch de etiquetas: torch.Size([64])  → (batch_size,)
Dtype textos:    torch.int64
Dtype etiquetas: torch.float32

Primer texto del batch (primeros 30 índices):
  [124, 113, 122, 65, 14, 2, 1222, 1, 158, 851, 8, 137, 178, 2691, 5101, 4946, 648, 1207, 116, 133, 98, 2537, 1836, 9, 112, 221, 1836, 1332, 193, 62]
Primeras 10 etiquetas del batch:
  [0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0]


In [30]:
# Matriz GloVe con vocabulario mejorado
if GLOVE_PATH.exists():
    embedding_matrix2 = load_glove(str(GLOVE_PATH), word2idx2, embed_dim=EMBED_DIM)

    print(f'Forma de la matriz de embeddings: {embedding_matrix2.shape}')
    print(f'  → {embedding_matrix2.shape[0]:,} palabras x {embedding_matrix2.shape[1]} dimensiones')
    print(f'Dtype: {embedding_matrix2.dtype}')
    print(f'\nVector de <PAD> (debe ser ceros): {embedding_matrix2[PAD_IDX][:10]}')

    word_ejemplo = 'the'
    if word_ejemplo in word2idx2:
        idx_ejemplo = word2idx2[word_ejemplo]
        print(f"Vector de '{word_ejemplo}' (primeros 10 dims): {embedding_matrix2[idx_ejemplo][:10].round(4)}")

GloVe cargado: 19,367 / 20,002 palabras encontradas (96.8% de cobertura)
Forma de la matriz de embeddings: (20002, 50)
  → 20,002 palabras x 50 dimensiones
Dtype: float64

Vector de <PAD> (debe ser ceros): [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Vector de 'the' (primeros 10 dims): [ 0.418   0.2497 -0.4124  0.1217  0.3453 -0.0445 -0.4969 -0.1786 -0.0007
 -0.6566]


In [31]:
# nn.Embedding con pesos mejorados
vocab_size2 = len(word2idx2)

embedding_layer2 = nn.Embedding(
    num_embeddings=vocab_size2,
    embedding_dim=EMBED_DIM,
    padding_idx=PAD_IDX,
)
embedding_layer2.weight = nn.Parameter(
    torch.tensor(embedding_matrix2, dtype=torch.float32)
)

print(f'Capa nn.Embedding mejorada:')
print(f'  Forma de pesos: {embedding_layer2.weight.shape}')
print(f'  padding_idx: {embedding_layer2.padding_idx}')

word_test = 'the'
if word_test in word2idx2:
    idx_test = word2idx2[word_test]
    vec_numpy2 = embedding_matrix2[idx_test][:5]
    with torch.no_grad():
        vec_pytorch2 = embedding_layer2(torch.tensor([idx_test], dtype=torch.long)).squeeze()[:5].numpy()
    print(f"\nVerificación para '{word_test}' (primeros 5 dims):")
    print(f'  Matriz numpy:   {vec_numpy2.round(4)}')
    print(f'  nn.Embedding:   {vec_pytorch2.round(4)}')
    print(f'  ¿Son iguales?   {__import__("numpy").allclose(vec_numpy2, vec_pytorch2)}')

with torch.no_grad():
    pad_vec2 = embedding_layer2(torch.tensor([PAD_IDX])).squeeze()
    print(f'\nVector de PAD (debe ser ceros): {pad_vec2[:10].tolist()}')

Capa nn.Embedding mejorada:
  Forma de pesos: torch.Size([20002, 50])
  padding_idx: 0

Verificación para 'the' (primeros 5 dims):
  Matriz numpy:   [ 0.418   0.2497 -0.4124  0.1217  0.3453]
  nn.Embedding:   [ 0.418   0.2497 -0.4124  0.1217  0.3453]
  ¿Son iguales?   True

Vector de PAD (debe ser ceros): [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [32]:
# Estadísticas de cobertura FINAL
glove_words2 = set()

if GLOVE_PATH.exists():
    with open(str(GLOVE_PATH), 'r', encoding='utf-8') as f:
        for line in f:
            word = line.split()[0]
            if word in word2idx2:
                glove_words2.add(word)

    total_vocab2 = len(word2idx2)
    con_glove2   = len(glove_words2)
    sin_glove2   = total_vocab2 - con_glove2

    print(f'=== Estadísticas de cobertura GloVe (pipeline mejorado) ===\n')
    print(f'Tamaño del vocabulario:             {total_vocab2:,}')
    print(f'Palabras CON vector GloVe:          {con_glove2:,} ({con_glove2/total_vocab2*100:.1f}%)')
    print(f'Palabras SIN vector GloVe (random): {sin_glove2:,} ({sin_glove2/total_vocab2*100:.1f}%)')

    palabras_sin_glove2 = [w for w in word2idx2 if w not in glove_words2 and w not in {'<PAD>', '<UNK>'}]
    if palabras_sin_glove2:
        print(f'\nEjemplos de palabras sin vector GloVe (primeras 20):')
        for w in palabras_sin_glove2[:20]:
            print(f"  - '{w}'")
else:
    print('GloVe no disponible.')

=== Estadísticas de cobertura GloVe (pipeline mejorado) ===

Tamaño del vocabulario:             20,002
Palabras CON vector GloVe:          19,367 (96.8%)
Palabras SIN vector GloVe (random): 635 (3.2%)

Ejemplos de palabras sin vector GloVe (primeras 20):
  - 'wikileak'
  - 'brexit'
  - 'что'
  - 'pende'
  - 'accorde'
  - 'daca'
  - 'isil'
  - 'reince'
  - 'puigdemont'
  - 'screengrab'
  - 'nusra'
  - 'daesh'
  - 'scaramucci'
  - 'somodevilla'
  - 'fjs'
  - 'это'
  - 'verita'
  - 'antifa'
  - 'bigote'
  - 'infowar'
